# HARS: Household Adaptive Risk-Scoring Framework
### Prototype Implementation (Part B)

This notebook implements the HARS framework as specified in Part A (Sections 6-7):
1. Nine-category device taxonomy
2. Five-factor weighted risk-scoring model
3. Plain-language output layer
4. Evidence-traceability mapping

No ML dependencies are used, consistent with the framework's design decision
(Table 6: "ML approaches infeasible for household deployment" — Hussain et al., 2020).

## 1. Setup

In [ ]:
import pandas as pd
from dataclasses import dataclass, field
from typing import Dict, List


## 2. Device Taxonomy (9 Categories)

Each category has baseline Device Vulnerability (DV) and Data Sensitivity (DS)
reference scores (1-10), used as a starting point that the per-factor rubrics
adjust based on the specific device's actual configuration.

In [ ]:
DEVICE_TAXONOMY = {
    "Entertainment & Media": {"baseline_dv": 5, "baseline_ds": 4,
        "examples": "Smart TVs, streaming devices"},
    "Voice Assistants & Smart Speakers": {"baseline_dv": 6, "baseline_ds": 7,
        "examples": "Amazon Echo, Google Nest"},
    "Home Security & Surveillance": {"baseline_dv": 7, "baseline_ds": 9,
        "examples": "Cameras, smart locks, alarms"},
    "Environmental Controls": {"baseline_dv": 5, "baseline_ds": 3,
        "examples": "Thermostats, HVAC controllers"},
    "Lighting Systems": {"baseline_dv": 4, "baseline_ds": 2,
        "examples": "Smart bulbs, switches"},
    "Kitchen & Appliances": {"baseline_dv": 5, "baseline_ds": 3,
        "examples": "Smart fridges, ovens"},
    "Health & Wellness Devices": {"baseline_dv": 6, "baseline_ds": 9,
        "examples": "Wearables, medical monitors"},
    "Networking Infrastructure": {"baseline_dv": 8, "baseline_ds": 6,
        "examples": "Routers, hubs, range extenders"},
    "Children\'s IoT & Toys": {"baseline_dv": 7, "baseline_ds": 8,
        "examples": "Smart toys, baby monitors"},
}

pd.DataFrame(DEVICE_TAXONOMY).T


## 3. Factor Weights (Table 6)

Weights are fixed and evidence-derived — not tunable by the user, to preserve
traceability to the literature synthesis.

In [ ]:
FACTOR_WEIGHTS = {
    "DV": 0.30,  # Device Vulnerability
    "DS": 0.25,  # Data Sensitivity
    "NE": 0.20,  # Network Exposure
    "UB": 0.15,  # User Behaviour
    "CS": 0.10,  # Compliance Status
}
assert sum(FACTOR_WEIGHTS.values()) == 1.0


## 4. Per-Factor Assessment Rubrics

Part A specified *that* structured assessment prompts exist, but not their
exact mechanics. These rubrics are the Week 1 design contribution: each
function takes simple, non-expert-answerable inputs and returns a 1-10 score.


In [ ]:
def assess_device_vulnerability(category: str, firmware_updated: bool,
                                 default_credentials: bool, device_age_years: float) -> int:
    """Rubric -> DV score (1-10). Higher = more vulnerable.
    Literature basis: Ali & Awad (2018) - device-tier deficiencies most prevalent."""
    score = DEVICE_TAXONOMY[category]["baseline_dv"]
    if not firmware_updated:
        score += 2
    if default_credentials:
        score += 2
    if device_age_years > 3:
        score += 1
    return max(1, min(10, round(score)))


def assess_data_sensitivity(category: str, handles_biometric_or_health: bool,
                             handles_financial: bool) -> int:
    """Rubric -> DS score (1-10).
    Literature basis: Emami-Naeini et al. (2021) - high-sensitivity data enables
    harm even from brief exposure."""
    score = DEVICE_TAXONOMY[category]["baseline_ds"]
    if handles_biometric_or_health:
        score += 2
    if handles_financial:
        score += 2
    return max(1, min(10, round(score)))


def assess_network_exposure(cloud_connected: bool, exposed_port_forwarding: bool,
                             uses_vpn_or_segmented_network: bool) -> int:
    """Rubric -> NE score (1-10).
    Literature basis: Aldahmani et al. (2023) - encrypted traffic still leaks
    sensitive metadata."""
    score = 3
    if cloud_connected:
        score += 3
    if exposed_port_forwarding:
        score += 3
    if uses_vpn_or_segmented_network:
        score -= 2
    return max(1, min(10, round(score)))


def assess_user_behaviour(changed_default_password: bool, updates_enabled: bool,
                           reuses_passwords: bool) -> int:
    """Rubric -> UB score (1-10). Higher = riskier behaviour.
    Literature basis: Emami-Naeini et al. (2021) - default credential retention
    is near-universal; update non-compliance is primary risk amplifier."""
    score = 2
    if not changed_default_password:
        score += 4
    if not updates_enabled:
        score += 3
    if reuses_passwords:
        score += 2
    return max(1, min(10, round(score)))


def assess_compliance_status(etsi_303645_certified: bool,
                              vendor_discloses_vulnerabilities: bool) -> int:
    """Rubric -> CS score (1-10). Higher = worse compliance (riskier).
    Literature basis: Heartfield et al. (2018) - ETSI EN 303 645 compliance
    correlates with baseline security posture."""
    score = 8
    if etsi_303645_certified:
        score -= 5
    if vendor_discloses_vulnerabilities:
        score -= 2
    return max(1, min(10, round(score)))


## 5. Scoring Engine

In [ ]:
def calculate_hars_score(dv: int, ds: int, ne: int, ub: int, cs: int) -> float:
    """HARS Score = (DV*0.30)+(DS*0.25)+(NE*0.20)+(UB*0.15)+(CS*0.10)  -- Section 7.1"""
    return round(
        dv * FACTOR_WEIGHTS["DV"] + ds * FACTOR_WEIGHTS["DS"] +
        ne * FACTOR_WEIGHTS["NE"] + ub * FACTOR_WEIGHTS["UB"] +
        cs * FACTOR_WEIGHTS["CS"], 2
    )


def get_risk_tier(score: float) -> str:
    if score <= 3:
        return "Low"
    elif score <= 6:
        return "Moderate"
    elif score <= 8:
        return "High"
    else:
        return "Critical"


## 6. Output Layer: Plain-Language Guidance

In [ ]:
TIER_GUIDANCE = {
    "Low": "Your household's smart devices show minimal risk. Continue routine "
           "practices: keep firmware updated and review device settings periodically.",
    "Moderate": "Some devices carry moderate risk. Change any default passwords, "
                "enable automatic updates, and review which devices connect to the internet directly.",
    "High": "Multiple devices show significant risk. Prioritise changing default "
            "credentials immediately, disable unnecessary remote/cloud access, and "
            "check for firmware updates on all flagged devices.",
    "Critical": "Immediate action recommended. One or more devices combine weak "
                "credentials, outdated firmware, and sensitive data handling. Isolate "
                "these devices on a separate network if possible and update or replace them.",
}

def get_guidance(tier: str) -> str:
    return TIER_GUIDANCE[tier]


## 7. Evidence Traceability

Every weight and factor is linked back to its literature source (Table 5/6),
printed alongside each result for auditability.

In [ ]:
TRACEABILITY = {
    "DV": "Ali & Awad (2018) - device-tier vulnerabilities most prevalent across household IoT categories",
    "DS": "Aldahmani et al. (2023); Emami-Naeini et al. (2021) - sensitivity determines breach impact severity",
    "NE": "Aldahmani et al. (2023); Mahmoud et al. (2015) - encrypted traffic still leaks metadata; cloud connectivity compounds exposure",
    "UB": "Emami-Naeini et al. (2021, 2019) - default credential retention and update non-compliance are primary risk amplifiers",
    "CS": "Heartfield et al. (2018); Nurse et al. (2017) - ETSI EN 303 645 compliance correlates with baseline security posture",
}


## 8. End-to-End Household Assessment (Non-Linear Aggregation)

**Week 2 refinement.** The original Week 1 version used a simple arithmetic
mean across devices, which under-represents risk: a single Critical device
got diluted by lower-risk devices. This revision reflects Table 5's finding
that aggregation amplifies risk non-linearly (Nguyen et al., 2026a) —
practically, one compromised device can expose the whole household network
("weakest link" effect).

**Two changes:**
1. **Quadratic mean (RMS)** replaces arithmetic mean — mathematically gives
   more weight to higher-risk devices instead of letting them average out.
2. **Critical-device floor** — if any single device is Critical, the
   household tier is floored at High, regardless of the aggregate score.

In [ ]:
import math

def aggregate_scores(device_scores: List[float]) -> float:
    """Quadratic mean (RMS) aggregation -- non-linear, weights high-risk
    devices more heavily than a simple average.
    Literature basis: Nguyen et al. (2026a), Table 5 -- aggregation amplifies
    risk non-linearly."""
    return round(math.sqrt(sum(s ** 2 for s in device_scores) / len(device_scores)), 2)


def run_household_assessment(devices: List[Dict]) -> Dict:
    """Constellation-level scoring across all enrolled devices."""
    results = []
    for d in devices:
        dv = assess_device_vulnerability(d["category"], d["firmware_updated"],
                                          d["default_credentials"], d["device_age_years"])
        ds = assess_data_sensitivity(d["category"], d["handles_biometric_or_health"],
                                      d["handles_financial"])
        ne = assess_network_exposure(d["cloud_connected"], d["exposed_port_forwarding"],
                                      d["uses_vpn_or_segmented_network"])
        ub = assess_user_behaviour(d["changed_default_password"], d["updates_enabled"],
                                    d["reuses_passwords"])
        cs = assess_compliance_status(d["etsi_303645_certified"],
                                       d["vendor_discloses_vulnerabilities"])
        score = calculate_hars_score(dv, ds, ne, ub, cs)
        tier = get_risk_tier(score)
        results.append({"device": d["name"], "category": d["category"],
                         "DV": dv, "DS": ds, "NE": ne, "UB": ub, "CS": cs,
                         "score": score, "tier": tier})

    device_scores = [r["score"] for r in results]

    # Non-linear aggregation (RMS)
    household_score = aggregate_scores(device_scores)
    household_tier = get_risk_tier(household_score)

    # Weakest-link floor: any Critical device floors household tier at High
    has_critical_device = any(r["tier"] == "Critical" for r in results)
    tier_order = ["Low", "Moderate", "High", "Critical"]
    if has_critical_device and tier_order.index(household_tier) < tier_order.index("High"):
        household_tier = "High"

    # For comparison/logging: what the old linear-mean approach would have given
    linear_mean_score = round(sum(device_scores) / len(device_scores), 2)

    return {
        "per_device": results,
        "household_score": household_score,
        "household_tier": household_tier,
        "guidance": get_guidance(household_tier),
        "linear_mean_score_for_comparison": linear_mean_score,
        "weakest_link_floor_applied": has_critical_device and household_tier != get_risk_tier(household_score),
    }


## 9. Test Scenarios

In [ ]:
test_household = [
    {"name": "Living Room Smart TV", "category": "Entertainment & Media",
     "firmware_updated": True, "default_credentials": False, "device_age_years": 1,
     "handles_biometric_or_health": False, "handles_financial": False,
     "cloud_connected": True, "exposed_port_forwarding": False, "uses_vpn_or_segmented_network": False,
     "changed_default_password": True, "updates_enabled": True, "reuses_passwords": False,
     "etsi_303645_certified": True, "vendor_discloses_vulnerabilities": True},

    {"name": "Front Door Camera", "category": "Home Security & Surveillance",
     "firmware_updated": False, "default_credentials": True, "device_age_years": 4,
     "handles_biometric_or_health": False, "handles_financial": False,
     "cloud_connected": True, "exposed_port_forwarding": True, "uses_vpn_or_segmented_network": False,
     "changed_default_password": False, "updates_enabled": False, "reuses_passwords": True,
     "etsi_303645_certified": False, "vendor_discloses_vulnerabilities": False},

    {"name": "Fitness Tracker", "category": "Health & Wellness Devices",
     "firmware_updated": True, "default_credentials": False, "device_age_years": 0.5,
     "handles_biometric_or_health": True, "handles_financial": False,
     "cloud_connected": True, "exposed_port_forwarding": False, "uses_vpn_or_segmented_network": True,
     "changed_default_password": True, "updates_enabled": True, "reuses_passwords": False,
     "etsi_303645_certified": True, "vendor_discloses_vulnerabilities": True},
]

results = run_household_assessment(test_household)

hh_score = results["household_score"]
hh_tier = results["household_tier"]
guidance_text = results["guidance"]
linear_score = results["linear_mean_score_for_comparison"]

print(f"Household HARS Score (RMS, non-linear): {hh_score}/10  ({hh_tier})")
print(f"Household HARS Score (old linear mean, for comparison): {linear_score}/10  ({get_risk_tier(linear_score)})")
wl_floor = results['weakest_link_floor_applied']
print(f"Weakest-link floor applied: {wl_floor}")
print(f"Guidance: {guidance_text}")
print()
pd.DataFrame(results["per_device"])


## 10. Validation Notes

Compare against Table 7 (Framework Evaluation Matrix) for your Week 1/2 log:
- **Explainable outputs** ✓ demonstrated (score + tier + plain-language guidance per device)
- **Aggregation-aware scoring** ✓ demonstrated (household score = constellation mean, not single-device)
- **Evidence traceability** ✓ demonstrated (TRACEABILITY dict, docstrings cite literature)
- **Lightweight, no-ML** ✓ demonstrated (pure arithmetic, pandas only)
- **Empirical validation with real households** ✗ not addressed here — synthetic test data only, consistent with Table 7's known limitation.

**Honest limitation to log:** the aggregation method used here (simple mean)
is a simplification. Table 5 cites non-linear risk amplification
(Nguyen et al., 2026a) — a true implementation might need a weighted or
non-linear aggregation function. Flag this as a refinement for a later week.

## 11. Stress Test: RMS vs. Linear Mean Divergence

This scenario isolates the effect of non-linear aggregation: a household with
**four well-configured, low-risk devices** and **one severely compromised
device** (outdated router, default credentials, exposed to the internet).

With a simple mean, four low scores would dilute the one dangerous device
toward Moderate — masking real risk. RMS should keep the household score
anchored near the dangerous device instead, consistent with the weakest-link
principle in Table 5.

In [ ]:
stress_household = [
    {"name": "Bedroom Smart Bulb", "category": "Lighting Systems",
     "firmware_updated": True, "default_credentials": False, "device_age_years": 0.5,
     "handles_biometric_or_health": False, "handles_financial": False,
     "cloud_connected": False, "exposed_port_forwarding": False, "uses_vpn_or_segmented_network": True,
     "changed_default_password": True, "updates_enabled": True, "reuses_passwords": False,
     "etsi_303645_certified": True, "vendor_discloses_vulnerabilities": True},

    {"name": "Kitchen Thermostat", "category": "Environmental Controls",
     "firmware_updated": True, "default_credentials": False, "device_age_years": 1,
     "handles_biometric_or_health": False, "handles_financial": False,
     "cloud_connected": False, "exposed_port_forwarding": False, "uses_vpn_or_segmented_network": True,
     "changed_default_password": True, "updates_enabled": True, "reuses_passwords": False,
     "etsi_303645_certified": True, "vendor_discloses_vulnerabilities": True},

    {"name": "Smart Plug", "category": "Kitchen & Appliances",
     "firmware_updated": True, "default_credentials": False, "device_age_years": 0.5,
     "handles_biometric_or_health": False, "handles_financial": False,
     "cloud_connected": False, "exposed_port_forwarding": False, "uses_vpn_or_segmented_network": True,
     "changed_default_password": True, "updates_enabled": True, "reuses_passwords": False,
     "etsi_303645_certified": True, "vendor_discloses_vulnerabilities": True},

    {"name": "Hallway Light Switch", "category": "Lighting Systems",
     "firmware_updated": True, "default_credentials": False, "device_age_years": 1,
     "handles_biometric_or_health": False, "handles_financial": False,
     "cloud_connected": False, "exposed_port_forwarding": False, "uses_vpn_or_segmented_network": True,
     "changed_default_password": True, "updates_enabled": True, "reuses_passwords": False,
     "etsi_303645_certified": True, "vendor_discloses_vulnerabilities": True},

    {"name": "Legacy Router", "category": "Networking Infrastructure",
     "firmware_updated": False, "default_credentials": True, "device_age_years": 6,
     "handles_biometric_or_health": False, "handles_financial": False,
     "cloud_connected": True, "exposed_port_forwarding": True, "uses_vpn_or_segmented_network": False,
     "changed_default_password": False, "updates_enabled": False, "reuses_passwords": True,
     "etsi_303645_certified": False, "vendor_discloses_vulnerabilities": False},
]

stress_results = run_household_assessment(stress_household)

rms_score = stress_results["household_score"]
rms_tier = stress_results["household_tier"]
mean_score = stress_results["linear_mean_score_for_comparison"]
mean_tier = get_risk_tier(mean_score)
floor_applied = stress_results["weakest_link_floor_applied"]

print("=== 4 low-risk devices + 1 severely compromised device ===\n")
print(f"Linear mean approach:  {mean_score}/10  ({mean_tier})")
print(f"RMS (non-linear):      {rms_score}/10  ({rms_tier})")
print(f"Weakest-link floor applied: {floor_applied}")
print()
print(f"Divergence: {round(rms_score - mean_score, 2)} points  "
      f"({'tier changed' if mean_tier != rms_tier else 'same tier, higher score'})")
print()
pd.DataFrame(stress_results["per_device"])
